In [50]:
import pandas as pd
import numpy as np

orig = pd.read_csv("/data/working/origination_v02.csv")

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_13924/55884575.py:4: DtypeWarning: Columns (25,26,30) have mixed types. Specify dtype option on import or set low_memory=False.
  orig = pd.read_csv("/Users/macbookpro/platform/Backend/data/processed/origination_v02.csv")


In [51]:
orig.isna().mean().round(6)

Unnamed: 0                         0.000000
CREDIT_SCORE                       0.000000
FIRST_TIME_HOMEBUYER_FLAG          0.000000
MSA                                0.143279
MI_PERCENTAGE                      0.000003
NUMBER_OF_UNITS                    0.000000
OCCUPANCY_STATUS                   0.000000
OCLTV                              0.000000
DTI                                0.000000
ORIGINAL_UPB                       0.000000
LTV                                0.000000
ORIGINAL_INTEREST_RATE             0.000000
CHANNEL                            0.000000
PPM_FLAG                           0.000000
PRODUCT_TYPE                       0.000000
STATE                              0.000000
PROPERTY_TYPE                      0.000000
POSTAL_CODE                        0.000000
LOAN_SEQUENCE_NUMBER               0.000000
LOAN_PURPOSE                       0.000000
ORIGINAL_LOAN_TERM                 0.000000
NUMBER_OF_BORROWERS                0.000000
SELLER_NAME                     

In [52]:
orig.head()

,Unnamed: 0,CREDIT_SCORE,FIRST_TIME_HOMEBUYER_FLAG,MSA,MI_PERCENTAGE,NUMBER_OF_UNITS,OCCUPANCY_STATUS,OCLTV,DTI,ORIGINAL_UPB,...,SERVICER_NAME,SUPER_CONFORMING_FLAG,PRE_RELIEF_REFI_LOAN_SEQ,PROGRAM_INDICATOR,RELIEF_REFINANCE_INDICATOR,PROPERTY_VALUATION_METHOD,IO_FLAG,MORTGAGE_INSURANCE_CANCELLATION,IS_MISSING_CREDIT_SCORE,IS_MISSING_DTI
0,0,706.0,N,20260.0,0.0,2,I,75.0,36.0,169000,...,Other servicers,N,NaN,9,N,7,N,9,0,0
1,1,615.0,N,36980.0,0.0,1,P,61.0,38.0,28000,...,Other servicers,N,NaN,9,N,7,N,9,0,0
2,2,681.0,Y,48424.0,30.0,1,P,95.0,46.0,271000,...,Other servicers,N,NaN,9,N,7,N,9,0,0
3,3,687.0,N,31860.0,0.0,1,P,39.0,44.0,75000,...,Other servicers,N,NaN,9,N,7,N,9,0,0
4,4,813.0,N,NaN,30.0,1,S,95.0,45.0,133000,...,U.S. BANK N.A.,N,NaN,9,N,7,N,9,0,0


In [53]:
orig['MSA'] = orig['MSA'].astype(str)
orig['MSA'] = orig['MSA'].fillna('RURAL')

In [54]:
orig = orig.drop(columns=['Unnamed: 0'])

In [55]:
orig.isna().mean().round(6)

CREDIT_SCORE                       0.000000
FIRST_TIME_HOMEBUYER_FLAG          0.000000
MSA                                0.000000
MI_PERCENTAGE                      0.000003
NUMBER_OF_UNITS                    0.000000
OCCUPANCY_STATUS                   0.000000
OCLTV                              0.000000
DTI                                0.000000
ORIGINAL_UPB                       0.000000
LTV                                0.000000
ORIGINAL_INTEREST_RATE             0.000000
CHANNEL                            0.000000
PPM_FLAG                           0.000000
PRODUCT_TYPE                       0.000000
STATE                              0.000000
PROPERTY_TYPE                      0.000000
POSTAL_CODE                        0.000000
LOAN_SEQUENCE_NUMBER               0.000000
LOAN_PURPOSE                       0.000000
ORIGINAL_LOAN_TERM                 0.000000
NUMBER_OF_BORROWERS                0.000000
SELLER_NAME                        0.000000
SERVICER_NAME                   

In [56]:
# Vérifier les cas
mask = orig['MI_PERCENTAGE'].isna()
orig[mask][['LOAN_SEQUENCE_NUMBER', 'LTV', 'MI_PERCENTAGE']]

,LOAN_SEQUENCE_NUMBER,LTV,MI_PERCENTAGE
349244,F10Q10047610,85.0,NaN
494555,F10Q10194131,85.0,NaN
540259,F10Q10240451,85.0,NaN
590029,F10Q10290474,123.0,NaN


In [57]:
# LTV <= 80 → MI_PERCENTAGE doit être 0
# LTV > 80 → MI_PERCENTAGE doit être > 0

orig[orig['MI_PERCENTAGE'].isna()]['LTV'].describe()

count      4.0
mean      94.5
std       19.0
min       85.0
25%       85.0
50%       85.0
75%       94.5
max      123.0
Name: LTV, dtype: float64

                La médiane du LTV est 70 et le 75ème percentile est 80 — donc la majorité des NaN ont un LTV ≤ 80.
                Ces prêts n'ont pas besoin d'assurance hypothécaire

In [58]:
# LTV <= 80 → MI_PERCENTAGE = 0
orig.loc[orig['MI_PERCENTAGE'].isna() & (orig['LTV'] <= 80), 'MI_PERCENTAGE'] = 0

# LTV > 80 → médiane par segment
orig.loc[orig['MI_PERCENTAGE'].isna() & (orig['LTV'] > 80), 'MI_PERCENTAGE'] = \
    orig[orig['LTV'] > 80]['MI_PERCENTAGE'].median()

# Vérification
orig['MI_PERCENTAGE'].isna().sum()

np.int64(0)

In [59]:
orig.isna().sum()[orig.isna().sum() > 0]

PRE_RELIEF_REFI_LOAN_SEQ    1292171
dtype: int64

Save data

In [60]:
orig.to_csv('/Users/macbookpro/platform/Backend/data/processed/orig_v03.csv', index= False)

In [48]:
def harmonize_dtypes(df):
    for col in df.columns:
        # Colonnes object (string mixte)
        if df[col].dtype == object:
            df[col] = df[col].astype(str).replace('nan', None)
        # Colonnes numériques → garder telles quelles
        # Colonnes datetime → garder telles quelles
    return df

orig = harmonize_dtypes(orig)
orig.to_parquet('/Users/macbookpro/platform/Backend/data/processed/origination_v03.parquet', index= False)
print("Origination sauvegardée ")

Origination sauvegardée 


In [49]:
test = pd.read_parquet('/data/raw/processed/origination_v03.parquet')
print(test.shape)
print(test.head())

(1390970, 32)
   CREDIT_SCORE FIRST_TIME_HOMEBUYER_FLAG      MSA  MI_PERCENTAGE  \
0         706.0                         N  20260.0            0.0   
1         615.0                         N  36980.0            0.0   
2         681.0                         Y  48424.0           30.0   
3         687.0                         N  31860.0            0.0   
4         813.0                         N     None           30.0   

   NUMBER_OF_UNITS OCCUPANCY_STATUS  OCLTV   DTI  ORIGINAL_UPB   LTV  ...  \
0                2                I   75.0  36.0        169000  75.0  ...   
1                1                P   61.0  38.0         28000  61.0  ...   
2                1                P   95.0  46.0        271000  95.0  ...   
3                1                P   39.0  44.0         75000  39.0  ...   
4                1                S   95.0  45.0        133000  95.0  ...   

     SERVICER_NAME SUPER_CONFORMING_FLAG PRE_RELIEF_REFI_LOAN_SEQ  \
0  Other servicers                     

In [61]:
data = pd.read_csv('/data/working/svcg_v01.csv')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_13924/2922892437.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/Users/macbookpro/platform/Backend/data/processed/svcg_v01.csv')


In [62]:
def harmonize_dtypes(df):
    for col in df.columns:
        # Colonnes object (string mixte)
        if df[col].dtype == object:
            df[col] = df[col].astype(str).replace('nan', None)
        # Colonnes numériques → garder telles quelles
        # Colonnes datetime → garder telles quelles
    return df


data = harmonize_dtypes(orig)
data.to_parquet('/Users/macbookpro/platform/Backend/data/processed/svcg_v01.parquet', index=False)
print("Origination sauvegardée ")

Origination sauvegardée 


In [63]:
test = pd.read_parquet('/data/raw/processed/svcg_v01.parquet')
print(test.shape)
print(test.head())

(1390970, 32)
   CREDIT_SCORE FIRST_TIME_HOMEBUYER_FLAG      MSA  MI_PERCENTAGE  \
0         706.0                         N  20260.0            0.0   
1         615.0                         N  36980.0            0.0   
2         681.0                         Y  48424.0           30.0   
3         687.0                         N  31860.0            0.0   
4         813.0                         N     None           30.0   

   NUMBER_OF_UNITS OCCUPANCY_STATUS  OCLTV   DTI  ORIGINAL_UPB   LTV  ...  \
0                2                I   75.0  36.0        169000  75.0  ...   
1                1                P   61.0  38.0         28000  61.0  ...   
2                1                P   95.0  46.0        271000  95.0  ...   
3                1                P   39.0  44.0         75000  39.0  ...   
4                1                S   95.0  45.0        133000  95.0  ...   

     SERVICER_NAME SUPER_CONFORMING_FLAG PRE_RELIEF_REFI_LOAN_SEQ  \
0  Other servicers                     